# Enzyme Function Classification Using Ensemble Learning

**Task:** Predict the first EC digit (classes 1–6) from amino-acid protein sequences.



## 1. Methodological overview

### Prediction target
The EC system is hierarchical. We intentionally predict only its first digit:

| Class | Enzyme family |
|---|---|
| 1 | Oxidoreductases |
| 2 | Transferases |
| 3 | Hydrolases |
| 4 | Lyases |
| 5 | Isomerases |
| 6 | Ligases |

For example, `EC:1.1.1.1 → 1` and `EC:3.5.4.4 → 3`.

### Representations
1. **AAC:** 20 normalized amino-acid frequencies.
2. **Dipeptide composition:** 400 normalized 2-mer frequencies.
3. **Tripeptide composition:** up to 8,000 normalized 3-mer frequencies, followed by `SelectKBest` for the classical ML models.
4. **SVD embedding:** the tripeptide sparse matrix compressed to 128 dimensions using `TruncatedSVD`. The vectorizer and SVD are fitted on training data only.
### Leakage prevention
All learned transformations—feature selection, SVD, and model fitting—are learned from training data only. Validation data is used for model/representation decisions; the test set is held back until final evaluation.

### Metrics
The primary metric is **Macro F1**. We additionally report **MCC, AUPRC, and accuracy**, as required. Macro F1 gives each class equal weight, which is important when class frequencies differ.

## 2. Colab setup and configuration

The default configuration is  reproducible. `TUNING_SAMPLE_SIZE` is a **runtime control**, not a change to the final dataset: by default, the final selected model can be refit on all available training rows. Set it to `None` if you want 5-fold tuning on the entire training split.

For a limited Colab runtime, a tuning sample can make the notebook practical while preserving the required 5-fold methodology. The notebook records the setting in the results metadata.

In [ ]:
# ============================================================
# GPU CHECK
# ============================================================

import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU enabled:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("GPU not available. Using CPU.")

print("Device:", device)

In [ ]:
import os
import sys
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
N_JOBS = 4  # Avoid oversubscribing a local machine

# ---- Local data and output locations ----
# The dataset is streamed from Hugging Face and cached locally.
DATA_PATH = Path("data") / "SwissProt-EC"  # Optional local Arrow export location
OUTPUT_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Dataset controls ----
MAX_ROWS = 1500                # Stratified cap to keep local training manageable
MIN_SEQUENCE_LENGTH = 5
REMOVE_DUPLICATE_SEQUENCES = True

# ---- Feature controls ----
TRIPEPTIDE_K = 100
SVD_COMPONENTS = 64

# ---- Tuning controls ----
# None = tune on the complete training split.
TUNING_SAMPLE_SIZE = 800
CV_FOLDS = 5  # Required by the project instructions

# ---- Optional tracks ----
RUN_SVD_EMBEDDING = False  # Optional; disable for a faster local run
RUN_SHAP = False  # Optional and relatively slow
USE_GPU = False # Set to True to enable GPU for LightGBM if runtime is GPU

# ---- Runtime / plotting ----
warnings.filterwarnings("ignore")
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print("Python:", sys.version)
print("Random state:", RANDOM_STATE)
print("Data path:", DATA_PATH)

In [ ]:
from datasets import load_dataset

DATASET_NAME = "DanielHesslow/SwissProt-EC"

# Stream a randomized, bounded slice instead of downloading all ~261K records.
# The result is cached in the Hugging Face cache under the local user profile.
stream = load_dataset(DATASET_NAME, split="train", streaming=True)
stream = stream.shuffle(seed=RANDOM_STATE, buffer_size=10000)
raw_limit = MAX_ROWS * 4
rows = list(stream.take(raw_limit))
df_raw = pd.DataFrame(rows)
print(f"Loaded {len(df_raw):,} streamed rows (pre-cleaning cap {raw_limit:,}).")
print(df_raw.head())


## 3. Install dependencies

`lightgbm` is used for the required gradient-boosting model and `shap` is used only in the optional interpretability section. All other core components are from the scientific Python/scikit-learn ecosystem.

In [ ]:
# Imports used throughout the notebook
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from lightgbm import LGBMClassifier

from scipy import sparse

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

print("Libraries loaded successfully.")


## 4. Load the SwissProt-EC data from Google Drive

The loader supports the common file formats you may get after downloading the dataset. It also supports a directory created with Hugging Face `Dataset.save_to_disk()`.

In [ ]:
# The dataset is streamed from Hugging Face in the preceding cell.
# This cell retains a local-file option for users with an exported Arrow dataset.
if DATA_PATH.exists() and list(DATA_PATH.glob("*.arrow")):
    from datasets import Dataset, concatenate_datasets

    arrow_files = sorted(DATA_PATH.glob("*.arrow"))
    local_shards = [Dataset.from_file(str(file)) for file in arrow_files]
    df_raw = concatenate_datasets(local_shards).to_pandas()
    if MAX_ROWS is not None and len(df_raw) > MAX_ROWS * 4:
        df_raw = df_raw.sample(MAX_ROWS * 4, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"Loaded local Arrow dataset: {df_raw.shape}")
else:
    print("Using the bounded Hugging Face stream loaded above.")

print("\nDATASET SUMMARY")
print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
display(df_raw.head())


## 5. Inspect the raw schema and normalize EC annotations

The exact physical column names can vary depending on how the Hugging Face dataset was exported. We therefore identify the sequence and EC-label columns conservatively and fail loudly if they cannot be found.

The project requires the **first digit** of an EC annotation as the target.

In [ ]:
# Inspect non-null examples to understand the exported schema
for col in df_raw.columns:
    example = df_raw[col].dropna().iloc[0] if df_raw[col].notna().any() else None
    print(f"{col:20s} | dtype={df_raw[col].dtype} | example={str(example)[:160]}")


## 6. Extract one EC class per protein

A protein can have hierarchical EC annotations. For this project, the target is the first EC digit. If a row contains several EC annotations, the extraction function returns the first valid top-level class.

Rows without a valid class in **1–6** are excluded because they cannot be used for the requested six-class task.

In [ ]:
import re
import numpy as np
import pandas as pd

# Valid first-level EC classes required by the project
VALID_CLASSES = {"1", "2", "3", "4", "5", "6"}


def first_ec_class(value):
    """
    Extract the first EC class (1–6) from an EC annotation.

    Examples
    --------
    'EC:2.7.1.30' -> 2
    'EC:3.-.-.-'  -> 3
    '1.1.1.1'     -> 1

    If multiple annotations are provided, the first valid
    EC class encountered is returned.
    """

    # Handle missing values
    if value is None or (
        isinstance(value, float) and np.isnan(value)
    ):
        return None

    # labels_str is normally a list of EC annotations,
    # but this also handles a single string.
    if isinstance(value, (list, tuple, set, np.ndarray, pd.Series)):
        values = list(value)
    else:
        values = [value]

    for item in values:

        if item is None:
            continue

        item = str(item).strip()

        # EC format: EC:2.7.1.30
        match = re.search(
            r"EC\s*:\s*([1-6])(?:\.|-|$)",
            item,
            flags=re.IGNORECASE
        )

        if match:
            return int(match.group(1))

        # Plain format: 2.7.1.30
        match = re.search(
            r"^\s*([1-6])(?:\.|-|$)",
            item
        )

        if match:
            return int(match.group(1))

    return None

# CREATE MODELING DATASET
df = pd.DataFrame({
    "sequence": df_raw["seq"].astype("string"),
    "ec_class": df_raw["labels_str"].apply(first_ec_class)
})


# CLEAN SEQUENCES
df["sequence"] = (
    df["sequence"]
    .str.upper()
    .str.strip()
)

df = df.dropna(
    subset=["sequence", "ec_class"]
).copy()

df["ec_class"] = df["ec_class"].astype(int)


# KEEP ONLY REQUIRED EC CLASSES
df = df[
    df["ec_class"].isin(
        {int(c) for c in VALID_CLASSES}
    )
].copy()


# INSPECT RESULT
print("=" * 60)
print("CLEANED DATASET")
print("=" * 60)

print("Rows with EC classes 1–6:", len(df))

print("\nClass distribution:")
print(
    df["ec_class"]
    .value_counts()
    .sort_index()
)

print("\nFirst five rows:")
display(df.head())


## 7. Data cleaning and quality control

Standard proteins are represented using the 20 canonical amino acids:

`A C D E F G H I K L M N P Q R S T V W Y`

We remove missing/empty sequences, sequences containing non-canonical symbols, and extremely short sequences. Duplicate sequences are handled **before splitting** to prevent identical sequences from appearing in both train and test sets. This is an important leakage-control step.

In [ ]:
VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

def sequence_is_valid(seq):
    return isinstance(seq, str) and len(seq) >= MIN_SEQUENCE_LENGTH and set(seq).issubset(VALID_AA)

df["length"] = df["sequence"].str.len()
df["valid_sequence"] = df["sequence"].map(sequence_is_valid)

quality = {
    "raw rows after EC extraction": len(df),
    "missing/empty or invalid sequences": int((~df["valid_sequence"]).sum()),
    "duplicate sequence rows": int(df["sequence"].duplicated(keep=False).sum()),
    "unique sequences": int(df["sequence"].nunique()),
}
display(pd.Series(quality, name="count"))

df = df[df["valid_sequence"]].drop(columns="valid_sequence").copy()

if REMOVE_DUPLICATE_SEQUENCES:
    # If duplicate sequences have conflicting top-level labels, inspect them first.
    conflict_counts = (
        df.groupby("sequence")["ec_class"]
        .nunique()
        .value_counts()
        .sort_index()
    )
    print("Number of distinct labels per duplicated sequence:")
    display(conflict_counts)

    # Keep one deterministic record per sequence.
    # Conflicting labels are rare/ambiguous; retaining one label avoids duplicate leakage.
    df = df.drop_duplicates(subset="sequence", keep="first").reset_index(drop=True)

print("Cleaned shape:", df.shape)
print("Class counts:")
display(df["ec_class"].value_counts().sort_index())


## 8. Exploratory Data Analysis



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

df["ec_class"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0]
)
axes[0].set_title("EC Class Distribution")
axes[0].set_xlabel("EC Class")
axes[0].set_ylabel("Number of sequences")

axes[1].hist(df["length"], bins=50)
axes[1].set_title("Protein Sequence Length Distribution")
axes[1].set_xlabel("Sequence length")
axes[1].set_ylabel("Number of sequences")

plt.tight_layout()
plt.show()

display(df.groupby("ec_class")["length"].describe().round(2))


In [ ]:
class_counts = df["ec_class"].value_counts().sort_index()
imbalance_ratio = class_counts.max() / class_counts.min()

print("Class proportions:")
display((class_counts / class_counts.sum()).rename("proportion").to_frame())
print(f"Majority/minority ratio: {imbalance_ratio:.2f}x")


## 9. Stratified 70/15/15 split

The required split is:

- **70% training**
- **15% validation**
- **15% test**

We first create a 70% training split and a 30% temporary split, then divide that temporary split equally into validation and test. Both operations are stratified. The test set remains untouched until final evaluation.

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["ec_class"],
    random_state=RANDOM_STATE
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["ec_class"],
    random_state=RANDOM_STATE
)

print(f"Train: {len(train_df):,} ({len(train_df)/len(df):.1%})")
print(f"Val:   {len(val_df):,} ({len(val_df)/len(df):.1%})")
print(f"Test:  {len(test_df):,} ({len(test_df)/len(df):.1%})")

split_summary = pd.DataFrame({
    "train": train_df["ec_class"].value_counts(normalize=True),
    "validation": val_df["ec_class"].value_counts(normalize=True),
    "test": test_df["ec_class"].value_counts(normalize=True),
}).sort_index()

display(split_summary.round(4))


# 10. Feature engineering

## 10.1 Amino Acid Composition (AAC)

AAC represents a sequence by the normalized frequency of each of the 20 canonical amino acids:

$$
AAC_a = \frac{\text{count}(a)}{L}
$$

where `L` is the sequence length.

This produces exactly **20 features** and captures global amino-acid composition, but it discards positional/order information. The assignment explicitly requires this representation.

In [ ]:
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")

def amino_acid_composition(sequences):
    X = np.zeros((len(sequences), len(AMINO_ACIDS)), dtype=np.float32)
    aa_to_idx = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

    for row, seq in enumerate(sequences):
        length = len(seq)
        counts = np.zeros(len(AMINO_ACIDS), dtype=np.float32)
        for aa in seq:
            counts[aa_to_idx[aa]] += 1.0
        X[row] = counts / length

    return X

X_train_aac = amino_acid_composition(train_df["sequence"].tolist())
X_val_aac   = amino_acid_composition(val_df["sequence"].tolist())
X_test_aac  = amino_acid_composition(test_df["sequence"].tolist())

print("AAC shapes:", X_train_aac.shape, X_val_aac.shape, X_test_aac.shape)
print("First row sums to:", X_train_aac[0].sum())


In [ ]:
# Wrap the AAC features in a pandas DataFrame for elegant displaying
aac_features_df = pd.DataFrame(X_train_aac, columns=AMINO_ACIDS)

# Display the first 5 rows with the amino acid headers
display(aac_features_df.head())

## 10.2 Dipeptide composition

Dipeptide composition counts adjacent pairs such as `AC`, `CD`, and `DE`.

There are:

$$
20^2 = 400
$$

possible dipeptides. We use a sparse character n-gram representation so the feature matrix does not need to be materialized as a large dense array. The representation is normalized to relative frequencies per sequence.

In [ ]:
def build_ngram_vectorizer(n):
    return CountVectorizer(
        analyzer="char",
        ngram_range=(n, n),
        lowercase=False,
        binary=False,
        dtype=np.float32
    )

def normalize_rows(X):
    row_sums = np.asarray(X.sum(axis=1)).ravel()
    row_sums[row_sums == 0] = 1.0
    return X.multiply((1.0 / row_sums)[:, None]).tocsr()

di_vec = build_ngram_vectorizer(2)
X_train_di = normalize_rows(di_vec.fit_transform(train_df["sequence"]))
X_val_di   = normalize_rows(di_vec.transform(val_df["sequence"]))
X_test_di  = normalize_rows(di_vec.transform(test_df["sequence"]))

print("Dipeptide feature dimension:", X_train_di.shape[1])
print("Shapes:", X_train_di.shape, X_val_di.shape, X_test_di.shape)


## 10.3 Tripeptide composition

Tripeptide composition captures local sequence order over three residues.

There are:

$$
20^3 = 8,000
$$

possible tripeptides. The raw matrix is sparse and high-dimensional. The project therefore requires feature selection or dimensionality reduction. We use both tracks:

1. `SelectKBest(mutual_info_classif)` for the classical tripeptide model track.
2. `TruncatedSVD(n_components=128)` for the optional embedding track.

The selector and SVD are fitted on training data only.

In [ ]:
tri_vec = build_ngram_vectorizer(3)

X_train_tri = normalize_rows(tri_vec.fit_transform(train_df["sequence"]))
X_val_tri   = normalize_rows(tri_vec.transform(val_df["sequence"]))
X_test_tri  = normalize_rows(tri_vec.transform(test_df["sequence"]))

print("Raw tripeptide feature dimension:", X_train_tri.shape[1])
print("Train non-zero entries:", X_train_tri.nnz)


### Tripeptide feature selection

`SelectKBest` is fitted only on the training split. Mutual information estimates how informative an individual feature is about the target. The exact `k` is a tunable modeling decision, so we expose it as a configuration parameter rather than presenting it as a universal biological truth.

In [ ]:
tri_selector = SelectKBest(
    score_func=mutual_info_classif,
    k=min(TRIPEPTIDE_K, X_train_tri.shape[1])
)

X_train_tri_sel = tri_selector.fit_transform(X_train_tri, train_df["ec_class"])
X_val_tri_sel   = tri_selector.transform(X_val_tri)
X_test_tri_sel  = tri_selector.transform(X_test_tri)

print("Selected tripeptide features:", X_train_tri_sel.shape[1])


## 10.4 Optional embedding representation: TruncatedSVD

Instead of selecting individual tripeptides, this track compresses the 8,000-dimensional sparse matrix into a lower-dimensional representation.

SVD learns a basis from the training matrix. Therefore `fit_transform` is used only for training and `transform` for validation/test. This is the leakage-safe procedure specified by the project.

In [ ]:
if RUN_SVD_EMBEDDING:
    svd = TruncatedSVD(
        n_components=SVD_COMPONENTS,
        random_state=RANDOM_STATE
    )

    X_train_svd = svd.fit_transform(X_train_tri)
    X_val_svd   = svd.transform(X_val_tri)
    X_test_svd  = svd.transform(X_test_tri)

    print("SVD embedding shapes:")
    print(X_train_svd.shape, X_val_svd.shape, X_test_svd.shape)
    print(f"Explained variance by {SVD_COMPONENTS} components: "
          f"{svd.explained_variance_ratio_.sum():.2%}")
else:
    X_train_svd = X_val_svd = X_test_svd = None


# 11. Modeling methodology

The three required base models are:

### Random Forest
An ensemble of decision trees trained with randomized sampling/feature selection. It can capture nonlinear relationships and provides a model-based feature-importance measure.

### LightGBM
A gradient-boosted decision-tree framework. Trees are built sequentially, with later trees attempting to correct errors made by earlier trees.

### SVM
A support-vector classifier that seeks a separating boundary with a margin. For this large sequence dataset, we use a **linear kernel** by default for practical scalability. We enable probabilities because soft voting and AUPRC/ROC analysis require class scores/probabilities.

### Imbalance strategy
The base models use `class_weight="balanced"` where supported. This changes the training loss/weighting without creating synthetic validation/test observations. The project permits class weighting as an imbalance strategy.

In [ ]:
# Common labels
y_train = train_df["ec_class"].to_numpy()
y_val   = val_df["ec_class"].to_numpy()
y_test  = test_df["ec_class"].to_numpy()

# A single model configuration can be reused across feature representations.
def make_models():
    rf = RandomForestClassifier(
        n_estimators=100, # Reduced for faster execution
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS
    )

    lgbm_params = {
        "objective": "multiclass",
        "num_class": 6,
        "n_estimators": 100, # Reduced for faster execution
        "class_weight": "balanced",
        "random_state": RANDOM_STATE,
        "n_jobs": N_JOBS,
        "verbosity": -1
    }
    if USE_GPU:
        lgbm_params["device"] = "gpu"

    lgbm = LGBMClassifier(**lgbm_params)

    svm = SVC(
        kernel="linear",
        probability=False,  # Probability calibration is added only for soft/stacked ensembles
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
    return rf, lgbm, svm

# 12. Evaluation utilities

The primary metric is Macro F1. We also calculate MCC, accuracy, ROC-AUC and AUPRC. For multiclass ROC/AUPRC, each class is evaluated in a one-vs-rest manner and macro-averaged.

This makes the evaluation less dependent on the largest class and exposes performance on minority classes.

In [ ]:
CLASS_ORDER = np.arange(1, 7)

def predict_scores(model, X):
    if callable(getattr(model, "predict_proba", None)):
        return model.predict_proba(X)

    if callable(getattr(model, "decision_function", None)):
        scores = np.asarray(model.decision_function(X))
        if scores.ndim == 1:
            scores = np.column_stack([-scores, scores])
        scores = scores - scores.max(axis=1, keepdims=True)
        exp_scores = np.exp(scores)
        return exp_scores / exp_scores.sum(axis=1, keepdims=True)

    # Hard VotingClassifier has no probability or decision score API.
    # Convert each estimator's class vote into a normalized vote fraction.
    if hasattr(model, "estimators_") and hasattr(model, "le_"):
        classes = np.asarray(model.classes_)
        votes = np.zeros((len(X), len(classes)), dtype=float)
        weights = getattr(model, "weights", None)
        if weights is None:
            weights = np.ones(len(model.estimators_), dtype=float)
        else:
            weights = np.asarray(weights, dtype=float)

        for estimator, weight in zip(model.estimators_, weights):
            encoded = np.asarray(estimator.predict(X), dtype=int)
            labels = model.le_.inverse_transform(encoded)
            for j, cls in enumerate(classes):
                votes[:, j] += weight * (labels == cls)
        total_weight = weights.sum()
        if total_weight > 0:
            return votes / total_weight

    # Last-resort scores for a predict-only classifier.
    classes = np.asarray(getattr(model, "classes_", CLASS_ORDER))
    pred = model.predict(X)
    scores = np.zeros((len(X), len(classes)), dtype=float)
    for j, cls in enumerate(classes):
        scores[:, j] = np.asarray(pred) == cls
    return scores

def evaluate_multiclass(model, X, y, split_name="validation"):
    pred = model.predict(X)
    proba = predict_scores(model, X)

    # Ensure probability columns correspond to classes 1..6.
    model_classes = getattr(model, "classes_", CLASS_ORDER)
    aligned = np.zeros((len(y), 6), dtype=float)
    for j, cls in enumerate(model_classes):
        if int(cls) in CLASS_ORDER:
            aligned[:, int(cls) - 1] = proba[:, j]

    metrics = {
        "split": split_name,
        "accuracy": accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro"),
        "mcc": matthews_corrcoef(y, pred),
        "roc_auc_macro_ovr": roc_auc_score(
            y, aligned, multi_class="ovr", average="macro"
        ),
        "auprc_macro_ovr": average_precision_score(
            pd.get_dummies(pd.Categorical(y, categories=CLASS_ORDER)).to_numpy(),
            aligned,
            average="macro"
        )
    }
    return metrics, pred, aligned

def metrics_table(records):
    return pd.DataFrame(records).set_index(["representation", "model", "split"]).sort_index()


# 13. Hyperparameter tuning

The project requires **5-fold cross-validation on the training data**. We use `RandomizedSearchCV` with five folds to explore a compact, meaningful search space.

The search is performed on the training split only. The validation and test sets are not used for fitting or tuning.

For practical Colab execution, `TUNING_SAMPLE_SIZE` can limit the number of training rows used during the search. If strict full-training tuning is desired, set it to `None`. After tuning, the best estimator is refit on the full training representation.

In [ ]:
from sklearn.base import clone
import joblib

def tuning_subset(X, y, sample_size=None):
    size = TUNING_SAMPLE_SIZE if sample_size is None else sample_size
    if size is None or len(y) <= size:
        return X, y

    idx, _ = train_test_split(
        np.arange(len(y)),
        train_size=size,
        stratify=y,
        random_state=RANDOM_STATE
    )
    return X[idx], y[idx]

cv = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

rf_param_dist = {
    "n_estimators": [60, 100],
    "max_depth": [None, 12, 20],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", 0.6]
}

lgbm_param_dist = {
    "n_estimators": [60, 100],
    "learning_rate": [0.05, 0.1],
    "num_leaves": [15, 31],
    "max_depth": [-1, 8],
    "min_child_samples": [20, 50],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.7, 0.9]
}

svm_param_dist = {
    "C": [0.1, 1.0, 3.0]
}

def tune_model(model, param_dist, X, y, n_iter=2, tuning_sample_size=None):
    X_tune, y_tune = tuning_subset(X, y, sample_size=tuning_sample_size)
    print(f"Tuning on {len(y_tune):,} stratified rows with {CV_FOLDS}-fold CV and {n_iter} candidates...")

    # Parallelize folds without nested estimator parallelism or repeated SVC calibration.
    search_model = clone(model)
    params = search_model.get_params(deep=False)
    if "n_jobs" in params:
        search_model.set_params(n_jobs=1)
    if "probability" in params:
        search_model.set_params(probability=False)

    search = RandomizedSearchCV(
        estimator=search_model,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring="f1_macro",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        verbose=1,
        refit=False
    )
    search.fit(X_tune, y_tune)

    # Refit only the winning configuration, using the full training split.
    best = clone(model).set_params(**search.best_params_)
    best.fit(X, y)
    return search, best


def save_training_checkpoint(stage):
    payload = {
        "stage": stage,
        "models_by_representation": globals().get("models_by_representation", {}),
        "ensemble_models": globals().get("ensemble_models", {}),
        "validation_records": globals().get("validation_records", []),
        "amino_acids": globals().get("AMINO_ACIDS"),
        "dipeptide_vectorizer": globals().get("di_vec"),
        "tripeptide_vectorizer": globals().get("tri_vec"),
        "tripeptide_selector": globals().get("tri_selector"),
        "random_state": RANDOM_STATE,
        "sample_cap": MAX_ROWS,
        "tuning_sample_size": TUNING_SAMPLE_SIZE,
        "cv_folds": CV_FOLDS,
    }
    path = OUTPUT_DIR / f"training_checkpoint_{stage}.joblib"
    joblib.dump(payload, path, compress=1)
    print(f"Saved checkpoint: {path.resolve()}")


# 14. Run the required models on AAC

AAC is small and dense, so it provides a useful baseline. The exact hyperparameters are selected by 5-fold Macro-F1 CV on training data.

In [ ]:
models_by_representation = {}
searches = {}
validation_records = []

representation_data = {
    "AAC": (X_train_aac, X_val_aac, X_test_aac),
}

rf, lgbm, svm = make_models()

search_rf_aac, best_rf_aac = tune_model(rf, rf_param_dist, X_train_aac, y_train)
search_lgbm_aac, best_lgbm_aac = tune_model(lgbm, lgbm_param_dist, X_train_aac, y_train)
search_svm_aac, best_svm_aac = tune_model(svm, svm_param_dist, X_train_aac, y_train)

models_by_representation["AAC"] = {
    "Random Forest": best_rf_aac,
    "LightGBM": best_lgbm_aac,
    "SVM": best_svm_aac
}
searches["AAC"] = {
    "Random Forest": search_rf_aac,
    "LightGBM": search_lgbm_aac,
    "SVM": search_svm_aac
}

for name, model in models_by_representation["AAC"].items():
    m, _, _ = evaluate_multiclass(model, X_val_aac, y_val, "validation")
    validation_records.append({"representation": "AAC", "model": name, **m})

display(pd.DataFrame(validation_records).sort_values("macro_f1", ascending=False))
save_training_checkpoint("AAC")


# 15. Dipeptide representation: required model comparison

Dipeptide features add local order information beyond global amino-acid composition. The same three models and metrics are reused so that the representation comparison is meaningful.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import RandomizedSearchCV, train_test_split

DIPEPTIDE_TUNING_SAMPLE_SIZE = min(600, len(y_train))

if DIPEPTIDE_TUNING_SAMPLE_SIZE < len(y_train):
    di_idx, _ = train_test_split(
        np.arange(len(y_train)),
        train_size=DIPEPTIDE_TUNING_SAMPLE_SIZE,
        stratify=y_train,
        random_state=RANDOM_STATE
    )
else:
    di_idx = np.arange(len(y_train))

X_di_tune = X_train_di[di_idx]
y_di_tune = y_train[di_idx]

def tune_dipeptide_model(model, param_dist):
    # Parallelize the CV fits, avoiding nested parallelism inside each estimator.
    search_model = clone(model)
    params = search_model.get_params(deep=False)

    if "n_jobs" in params:
        search_model.set_params(n_jobs=1)
    if "probability" in params:
        # SVC probability calibration is costly inside every CV fit.
        search_model.set_params(probability=False)

    search = RandomizedSearchCV(
        estimator=search_model,
        param_distributions=param_dist,
        n_iter=2,
        scoring="f1_macro",
        cv=cv,                   # Uses the existing five-fold stratified CV
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        verbose=1,
        refit=False
    )
    search.fit(X_di_tune, y_di_tune)

    # Refit the winning settings on all available dipeptide training rows.
    best_model = clone(model).set_params(**search.best_params_)
    best_model.fit(X_train_di, y_train)
    return search, best_model


rf, lgbm, svm = make_models()

search_rf_di, best_rf_di = tune_dipeptide_model(rf, rf_param_dist)
search_lgbm_di, best_lgbm_di = tune_dipeptide_model(lgbm, lgbm_param_dist)
search_svm_di, best_svm_di = tune_dipeptide_model(svm, svm_param_dist)

models_by_representation["Dipeptide"] = {
    "Random Forest": best_rf_di,
    "LightGBM": best_lgbm_di,
    "SVM": best_svm_di
}
searches["Dipeptide"] = {
    "Random Forest": search_rf_di,
    "LightGBM": search_lgbm_di,
    "SVM": search_svm_di
}

for name, model in models_by_representation["Dipeptide"].items():
    metrics, _, _ = evaluate_multiclass(
        model, X_val_di, y_val, "validation"
    )
    validation_records.append({
        "representation": "Dipeptide",
        "model": name,
        **metrics
    })

display(
    pd.DataFrame(validation_records)
    .sort_values("macro_f1", ascending=False)
)
save_training_checkpoint("dipeptide")


# 16. Tripeptide representation: required model comparison

The raw tripeptide space contains up to 8,000 features. We use the training-fitted mutual-information selector to retain `TRIPEPTIDE_K` features before model fitting.

In [ ]:
rf, lgbm, svm = make_models()

search_rf_tri, best_rf_tri = tune_model(rf, rf_param_dist, X_train_tri_sel, y_train)
search_lgbm_tri, best_lgbm_tri = tune_model(lgbm, lgbm_param_dist, X_train_tri_sel, y_train)
search_svm_tri, best_svm_tri = tune_model(svm, svm_param_dist, X_train_tri_sel, y_train)

models_by_representation["Tripeptide-Selected"] = {
    "Random Forest": best_rf_tri,
    "LightGBM": best_lgbm_tri,
    "SVM": best_svm_tri
}
searches["Tripeptide-Selected"] = {
    "Random Forest": search_rf_tri,
    "LightGBM": search_lgbm_tri,
    "SVM": search_svm_tri
}

for name, model in models_by_representation["Tripeptide-Selected"].items():
    m, _, _ = evaluate_multiclass(model, X_val_tri_sel, y_val, "validation")
    validation_records.append({"representation": "Tripeptide-Selected", "model": name, **m})

validation_df = pd.DataFrame(validation_records).sort_values(
    ["macro_f1", "mcc"], ascending=False
)
display(validation_df)
save_training_checkpoint("tripeptide")


# 17. Optional embedding track

The SVD embedding is a different representation of the same tripeptide counts. We reuse the **same model families and metrics**, as required by the assignment, so that differences can be attributed to representation rather than a completely different modeling setup. fileciteturn0file0L990-L1006

In [ ]:
if RUN_SVD_EMBEDDING:
    rf, lgbm, svm = make_models()

    search_rf_svd, best_rf_svd = tune_model(rf, rf_param_dist, X_train_svd, y_train)
    search_lgbm_svd, best_lgbm_svd = tune_model(lgbm, lgbm_param_dist, X_train_svd, y_train)
    search_svm_svd, best_svm_svd = tune_model(svm, svm_param_dist, X_train_svd, y_train)

    models_by_representation["Tripeptide-SVD-128"] = {
        "Random Forest": best_rf_svd,
        "LightGBM": best_lgbm_svd,
        "SVM": best_svm_svd
    }
    searches["Tripeptide-SVD-128"] = {
        "Random Forest": search_rf_svd,
        "LightGBM": search_lgbm_svd,
        "SVM": search_svm_svd
    }

    for name, model in models_by_representation["Tripeptide-SVD-128"].items():
        m, _, _ = evaluate_multiclass(model, X_val_svd, y_val, "validation")
        validation_records.append({"representation": "Tripeptide-SVD-128", "model": name, **m})

    validation_df = pd.DataFrame(validation_records).sort_values(
        ["macro_f1", "mcc"], ascending=False
    )
    display(validation_df)


# 18. Select the representation/model family for ensemble construction

We choose the strongest **validation Macro F1** result as the representation for the final ensemble. This decision is made using validation data only; the test set remains untouched.

For a fair ensemble, the three required base learners use the **same selected representation**. This also makes hard voting, soft voting, and stacking straightforward and interpretable.

In [ ]:
# Map representation names to their train/validation/test matrices.
feature_matrices = {
    "AAC": (X_train_aac, X_val_aac, X_test_aac),
    "Dipeptide": (X_train_di, X_val_di, X_test_di),
    "Tripeptide-Selected": (X_train_tri_sel, X_val_tri_sel, X_test_tri_sel),
}

if RUN_SVD_EMBEDDING:
    feature_matrices["Tripeptide-SVD-128"] = (X_train_svd, X_val_svd, X_test_svd)

# Select the best representation according to the best of its three base models.
rep_summary = (
    validation_df.groupby("representation")["macro_f1"]
    .max()
    .sort_values(ascending=False)
)

print("Best validation Macro F1 by representation:")
display(rep_summary.to_frame("best_validation_macro_f1"))

SELECTED_REPRESENTATION = rep_summary.index[0]
print("Selected representation:", SELECTED_REPRESENTATION)

X_train_final, X_val_final, X_test_final = feature_matrices[SELECTED_REPRESENTATION]


# 19. Required voting ensembles

The project requires:

1. **Hard voting** — majority class vote.
2. **Soft voting** — average predicted class probabilities.
3. **Stacking** — base-model predictions are combined by a logistic-regression meta-classifier with `class_weight="balanced"` and 5-fold internal CV.

The stacking meta-classifier is deliberately **Logistic Regression**, not an MLP, following the specification. fileciteturn0file0L1073-L1112

In [ ]:
# Reuse the tuned best configurations for the selected representation.
base_models = models_by_representation[SELECTED_REPRESENTATION]

rf_base = base_models["Random Forest"]
lgbm_base = base_models["LightGBM"]
svm_base = base_models["SVM"]

# Clone estimators so each ensemble fits its own copy.
from sklearn.base import clone

rf_e = clone(rf_base)
lgbm_e = clone(lgbm_base)
svm_e = clone(svm_base)

hard_voting = VotingClassifier(
    estimators=[
        ("rf", rf_e),
        ("lgbm", lgbm_e),
        ("svm", svm_e)
    ],
    voting="hard",
    n_jobs=N_JOBS
)

rf_e2 = clone(rf_base)
lgbm_e2 = clone(lgbm_base)
svm_e2 = CalibratedClassifierCV(estimator=clone(svm_base), cv=3)

soft_voting = VotingClassifier(
    estimators=[
        ("rf", rf_e2),
        ("lgbm", lgbm_e2),
        ("svm", svm_e2)
    ],
    voting="soft",
    n_jobs=N_JOBS
)

rf_e3 = clone(rf_base)
lgbm_e3 = clone(lgbm_base)
svm_e3 = CalibratedClassifierCV(estimator=clone(svm_base), cv=3)

stacking = StackingClassifier(
    estimators=[
        ("rf", rf_e3),
        ("lgbm", lgbm_e3),
        ("svm", svm_e3)
    ],
    final_estimator=LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=RANDOM_STATE
    ),
    cv=5,
    stack_method="predict_proba",
    n_jobs=N_JOBS
)

ensemble_models = {
    "Hard Voting": hard_voting,
    "Soft Voting": soft_voting,
    "Stacking": stacking
}

ensemble_validation_records = []

for name, model in ensemble_models.items():
    print(f"Fitting {name}...")
    if not hasattr(model, "estimators_"):
        model.fit(X_train_final, y_train)
    m, _, _ = evaluate_multiclass(model, X_val_final, y_val, "validation")
    ensemble_validation_records.append({
        "representation": SELECTED_REPRESENTATION,
        "model": name,
        **m
    })

ensemble_validation_df = pd.DataFrame(ensemble_validation_records)
display(ensemble_validation_df.sort_values("macro_f1", ascending=False))
save_training_checkpoint("ensembles")


# 20. Final test evaluation

Only after all representation/model/ensemble decisions have been made do we evaluate on the held-out test set.

We include all three individual models from the selected representation plus the three ensembles. This provides the required comparison of individual models versus ensemble methods.

In [ ]:
final_models = {
    **base_models,
    **ensemble_models
}

test_records = []
test_predictions = {}
test_probabilities = {}

for name, model in final_models.items():
    m, pred, proba = evaluate_multiclass(
        model, X_test_final, y_test, "test"
    )
    test_records.append({
        "representation": SELECTED_REPRESENTATION,
        "model": name,
        **m
    })
    test_predictions[name] = pred
    test_probabilities[name] = proba

test_results = pd.DataFrame(test_records).sort_values(
    ["macro_f1", "mcc"], ascending=False
)

display(test_results)


## 21. Per-class precision, recall and F1

The project explicitly requires a per-class table with precision, recall and F1. We report these for the selected final model.

**Important:** "easiest" and "hardest" below are defined descriptively from observed test metrics, not as biological judgments.

In [ ]:
# Select final model by validation Macro F1, not by test performance.
ensemble_val_lookup = ensemble_validation_df.set_index("model")["macro_f1"]
base_val_lookup = validation_df[
    validation_df["representation"] == SELECTED_REPRESENTATION
].set_index("model")["macro_f1"]

candidate_validation = pd.concat([base_val_lookup, ensemble_val_lookup])
FINAL_MODEL_NAME = candidate_validation.idxmax()
FINAL_MODEL = final_models[FINAL_MODEL_NAME]

joblib.dump({
    "model": FINAL_MODEL,
    "model_name": FINAL_MODEL_NAME,
    "representation": SELECTED_REPRESENTATION,
    "class_order": CLASS_ORDER.tolist(),
    "amino_acids": AMINO_ACIDS,
    "dipeptide_vectorizer": di_vec if SELECTED_REPRESENTATION == "Dipeptide" else None,
    "tripeptide_vectorizer": tri_vec if SELECTED_REPRESENTATION == "Tripeptide-Selected" else None,
    "tripeptide_selector": tri_selector if SELECTED_REPRESENTATION == "Tripeptide-Selected" else None,
}, OUTPUT_DIR / "selected_model_checkpoint.joblib", compress=1)
print("Saved selected-model checkpoint before report plots.")

print("Final model selected from validation:", FINAL_MODEL_NAME)

final_pred = test_predictions[FINAL_MODEL_NAME]
final_proba = test_probabilities[FINAL_MODEL_NAME]

report = classification_report(
    y_test,
    final_pred,
    labels=CLASS_ORDER,
    target_names=[f"EC {c}" for c in CLASS_ORDER],
    output_dict=True,
    zero_division=0
)

per_class = pd.DataFrame({
    "Class": [f"EC {c}" for c in CLASS_ORDER],
    "Precision": [report[f"EC {c}"]["precision"] for c in CLASS_ORDER],
    "Recall": [report[f"EC {c}"]["recall"] for c in CLASS_ORDER],
    "F1": [report[f"EC {c}"]["f1-score"] for c in CLASS_ORDER],
    "Support": [int(report[f"EC {c}"]["support"]) for c in CLASS_ORDER],
})

display(per_class.round(4))


# 22. Normalized confusion matrix

The assignment requests a confusion matrix **normalized by true class**. Each row therefore answers:

> Of the proteins whose true class is X, what fraction were predicted as each class?

In [ ]:
cm = confusion_matrix(y_test, final_pred, labels=CLASS_ORDER, normalize="true")

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt=".2f",
    xticklabels=[f"EC {c}" for c in CLASS_ORDER],
    yticklabels=[f"EC {c}" for c in CLASS_ORDER],
)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title(f"Normalized Confusion Matrix — {FINAL_MODEL_NAME}")
plt.tight_layout()
plt.show()


# 23. One-vs-rest ROC curves

For each EC class, we construct a one-vs-rest target:

`class X` vs `all other classes`.

We then plot the ROC curve and calculate AUC. This follows the requested multiclass ROC analysis. fileciteturn0file0L1155-L1159

In [ ]:
y_test_onehot = pd.get_dummies(
    pd.Categorical(y_test, categories=CLASS_ORDER)
).to_numpy()

plt.figure(figsize=(9, 7))

roc_auc_by_class = {}

for i, cls in enumerate(CLASS_ORDER):
    fpr, tpr, _ = roc_curve(y_test_onehot[:, i], final_proba[:, i])
    class_auc = auc(fpr, tpr)
    roc_auc_by_class[cls] = class_auc
    plt.plot(fpr, tpr, label=f"EC {cls} (AUC={class_auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"One-vs-Rest ROC Curves — {FINAL_MODEL_NAME}")
plt.legend()
plt.tight_layout()
plt.show()

display(pd.DataFrame({
    "EC Class": [f"EC {c}" for c in CLASS_ORDER],
    "ROC-AUC": [roc_auc_by_class[c] for c in CLASS_ORDER]
}))


# 24. One-vs-rest Precision–Recall curves

AUPRC is especially informative when positive examples are relatively rare. For each class, we compute the one-vs-rest precision–recall curve and average precision.

In [ ]:
plt.figure(figsize=(9, 7))

auprc_by_class = {}

for i, cls in enumerate(CLASS_ORDER):
    precision, recall, _ = precision_recall_curve(
        y_test_onehot[:, i],
        final_proba[:, i]
    )
    ap = average_precision_score(
        y_test_onehot[:, i],
        final_proba[:, i]
    )
    auprc_by_class[cls] = ap
    plt.plot(recall, precision, label=f"EC {cls} (AP={ap:.3f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"One-vs-Rest Precision–Recall Curves — {FINAL_MODEL_NAME}")
plt.legend()
plt.tight_layout()
plt.show()

display(pd.DataFrame({
    "EC Class": [f"EC {c}" for c in CLASS_ORDER],
    "AUPRC": [auprc_by_class[c] for c in CLASS_ORDER]
}))


# 25. Feature importance

For tree models, impurity-based feature importance is available directly through `feature_importances_`.

We prefer **LightGBM or Random Forest** as requested. If the selected final model is an ensemble, we inspect the LightGBM component from the selected representation.

Interpretation must remain cautious:

> An important feature is associated with predictive performance; it is not automatically a causal biological mechanism.

This distinction is explicitly emphasized by the assignment. fileciteturn0file0L1138-L1174

In [ ]:
importance_model = base_models["LightGBM"]

if hasattr(importance_model, "feature_importances_"):
    importances = np.asarray(importance_model.feature_importances_)

    if SELECTED_REPRESENTATION == "AAC":
        feature_names = np.array(AMINO_ACIDS)
    elif SELECTED_REPRESENTATION == "Dipeptide":
        feature_names = np.array(di_vec.get_feature_names_out())
    elif SELECTED_REPRESENTATION == "Tripeptide-Selected":
        all_names = np.array(tri_vec.get_feature_names_out())
        selected_mask = tri_selector.get_support()
        feature_names = all_names[selected_mask]
    elif SELECTED_REPRESENTATION == "Tripeptide-SVD-128":
        feature_names = np.array([f"SVD_{i+1:03d}" for i in range(len(importances))])
    else:
        feature_names = np.array([f"feature_{i}" for i in range(len(importances))])

    fi = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values("importance", ascending=False)

    display(fi.head(25))

    plt.figure(figsize=(10, 7))
    top = fi.head(20).sort_values("importance")
    plt.barh(top["feature"], top["importance"])
    plt.xlabel("LightGBM feature importance")
    plt.ylabel("Feature")
    plt.title(f"Top Features — {SELECTED_REPRESENTATION}")
    plt.tight_layout()
    plt.show()
else:
    print("The selected LightGBM model does not expose feature_importances_.")


# 26. Error analysis

Error analysis is not simply a list of incorrect predictions. We identify:

- classes with the strongest/weakest observed F1;
- the largest off-diagonal confusion pairs;
- examples where the final model is uncertain;
- whether ensemble methods improved over individual models.

Any biological explanation is treated as a **hypothesis**, not a demonstrated mechanism. The assignment explicitly asks us to separate observed results from possible explanations. fileciteturn0file0L857-L882

In [ ]:
# Per-class observed difficulty
difficulty = per_class[["Class", "Precision", "Recall", "F1", "Support"]].copy()
print("Observed per-class performance:")
display(difficulty.sort_values("F1"))

hardest_class = difficulty.loc[difficulty["F1"].idxmin(), "Class"]
easiest_class = difficulty.loc[difficulty["F1"].idxmax(), "Class"]

print(f"Highest observed test F1: {easiest_class}")
print(f"Lowest observed test F1:  {hardest_class}")

# Largest confusion pairs, excluding diagonal
cm_counts = confusion_matrix(y_test, final_pred, labels=CLASS_ORDER)
pairs = []
for i, true_cls in enumerate(CLASS_ORDER):
    for j, pred_cls in enumerate(CLASS_ORDER):
        if i != j:
            pairs.append({
                "True": true_cls,
                "Predicted": pred_cls,
                "Count": cm_counts[i, j]
            })

confusion_pairs = pd.DataFrame(pairs).sort_values("Count", ascending=False)
print("Most frequent observed confusion pairs:")
display(confusion_pairs.head(10))


In [ ]:
# Inspect uncertain predictions: low maximum predicted probability.
uncertainty = pd.DataFrame({
    "sequence": test_df["sequence"].to_numpy(),
    "true_class": y_test,
    "predicted_class": final_pred,
    "confidence": final_proba.max(axis=1)
})

uncertain_examples = uncertainty.sort_values("confidence").head(20)
display(uncertain_examples)


## Ensemble comparison

The question is not whether an ensemble is automatically better. We compare the observed validation/test Macro F1, MCC, AUPRC and accuracy and report what happened.

This avoids assuming that combining models must improve performance.

In [ ]:
comparison_rows = []

# Individual selected-representation models
for name, model in base_models.items():
    comparison_rows.append({
        "model": name,
        "validation_macro_f1": base_val_lookup[name],
        "test_macro_f1": test_results.loc[test_results["model"] == name, "macro_f1"].iloc[0],
        "test_mcc": test_results.loc[test_results["model"] == name, "mcc"].iloc[0],
        "test_auprc": test_results.loc[test_results["model"] == name, "auprc_macro_ovr"].iloc[0],
        "test_accuracy": test_results.loc[test_results["model"] == name, "accuracy"].iloc[0],
    })

for name in ensemble_models:
    comparison_rows.append({
        "model": name,
        "validation_macro_f1": ensemble_val_lookup[name],
        "test_macro_f1": test_results.loc[test_results["model"] == name, "macro_f1"].iloc[0],
        "test_mcc": test_results.loc[test_results["model"] == name, "mcc"].iloc[0],
        "test_auprc": test_results.loc[test_results["model"] == name, "auprc_macro_ovr"].iloc[0],
        "test_accuracy": test_results.loc[test_results["model"] == name, "accuracy"].iloc[0],
    })

ensemble_comparison = pd.DataFrame(comparison_rows)
display(ensemble_comparison.sort_values("validation_macro_f1", ascending=False))


# 27. Optional SHAP analysis

SHAP provides local additive explanations of model predictions. For tree models, `TreeExplainer` is the appropriate family of explainer.

Because the full dataset and high-dimensional sequence representations can be expensive to explain, this section intentionally uses a **small test subset**. The subset is only for explanation; it does not affect model fitting or evaluation.

For multiclass LightGBM, SHAP returns class-specific contributions. We summarize mean absolute SHAP values across classes and examples to identify globally influential features.

In [ ]:
if RUN_SHAP:
    import shap

    # Explain the LightGBM base model using a small held-out subset.
    SHAP_SAMPLE_SIZE = min(500, len(test_df))
    shap_rng = np.random.RandomState(RANDOM_STATE)
    shap_idx = shap_rng.choice(len(test_df), size=SHAP_SAMPLE_SIZE, replace=False)

    X_shap = X_test_final[shap_idx]
    y_shap = y_test[shap_idx]

    # TreeExplainer works directly with the fitted LightGBM model.
    shap_explainer = shap.TreeExplainer(importance_model)
    shap_values = shap_explainer.shap_values(X_shap)

    # Handle SHAP's multiclass return conventions.
    if isinstance(shap_values, list):
        abs_mean = np.mean(
            np.stack([np.abs(v) for v in shap_values], axis=0),
            axis=(0, 1)
        )
    else:
        arr = np.asarray(shap_values)
        if arr.ndim == 3:
            # Possible layouts: (n_samples, n_features, n_classes)
            abs_mean = np.abs(arr).mean(axis=(0, 2))
        else:
            abs_mean = np.abs(arr).mean(axis=0)

    shap_summary = pd.DataFrame({
        "feature": feature_names,
        "mean_abs_shap": abs_mean
    }).sort_values("mean_abs_shap", ascending=False)

    display(shap_summary.head(25))

    plt.figure(figsize=(10, 7))
    top_shap = shap_summary.head(20).sort_values("mean_abs_shap")
    plt.barh(top_shap["feature"], top_shap["mean_abs_shap"])
    plt.xlabel("Mean absolute SHAP value")
    plt.ylabel("Feature")
    plt.title("Global SHAP Feature Importance")
    plt.tight_layout()
    plt.show()
else:
    print("SHAP section disabled.")


# 28. Final results summary

This table is the compact result that should feed the written report and presentation.

**Do not select a final model using test performance.** The final model was selected using validation Macro F1; the test results below are a final held-out estimate.

In [ ]:
final_summary = test_results.copy()

# Add representation/model-selection metadata.
final_summary["selected_representation"] = SELECTED_REPRESENTATION
final_summary["final_model"] = final_summary["model"].eq(FINAL_MODEL_NAME)

display(
    final_summary[
        [
            "model",
            "selected_representation",
            "final_model",
            "macro_f1",
            "mcc",
            "auprc_macro_ovr",
            "accuracy",
            "roc_auc_macro_ovr"
        ]
    ].round(4)
)


# 29. Reproducibility record



In [ ]:
reproducibility_record = {
    "random_state": RANDOM_STATE,
    "split": {"train": 0.70, "validation": 0.15, "test": 0.15},
    "duplicate_removal": REMOVE_DUPLICATE_SEQUENCES,
    "min_sequence_length": MIN_SEQUENCE_LENGTH,
    "tripeptide_selected_k": TRIPEPTIDE_K,
    "svd_components": SVD_COMPONENTS if RUN_SVD_EMBEDDING else None,
    "tuning_sample_size": TUNING_SAMPLE_SIZE,
    "dipeptide_tuning_sample_size": DIPEPTIDE_TUNING_SAMPLE_SIZE,
    "cv_folds": CV_FOLDS,
    "selected_representation": SELECTED_REPRESENTATION,
    "final_model": FINAL_MODEL_NAME,
    "dataset_rows_after_cleaning": len(df),
    "data_source": DATASET_NAME,
    "sample_cap": MAX_ROWS,
    "optional_svd_enabled": RUN_SVD_EMBEDDING,
    "optional_shap_enabled": RUN_SHAP
}

print(json.dumps(reproducibility_record, indent=2))


# 30. Methodology discussion
Use the actual measured results above to replace the bracketed values.

### Introduction
We formulated enzyme-function prediction as a supervised six-class classification problem using the first digit of the EC number as the target.

### Dataset and preprocessing
The SwissProt-EC dataset was loaded from Google Drive. Missing/invalid sequences were removed, canonical amino-acid symbols were enforced, and duplicate sequences were removed before splitting to reduce leakage risk.

### Feature engineering
Three required representations were implemented: amino-acid composition (20 features), dipeptide composition (400 features), and tripeptide composition (up to 8,000 features). The tripeptide representation was reduced using training-only mutual-information feature selection. An additional optional representation compressed the tripeptide matrix to 128 dimensions with training-only TruncatedSVD.

### Modeling
Random Forest, LightGBM and a linear SVM were tuned using five-fold stratified cross-validation on training data. Class weighting was used to address imbalance without modifying validation/test distributions.

### Ensemble learning
Hard voting, soft voting and stacking were implemented. The stacking meta-classifier was logistic regression with balanced class weights and five-fold internal cross-validation.

### Evaluation
Macro F1 was treated as the primary metric, with MCC, AUPRC, accuracy and one-vs-rest ROC-AUC reported as additional measures. A normalized confusion matrix and per-class precision/recall/F1 were used for detailed analysis.

### Interpretation and limitations
Feature importance and optional SHAP analysis identify predictive associations, not causal biological mechanisms. Errors may also reflect limitations of sequence-only representations, class imbalance, label ambiguity, evolutionary redundancy, and the fact that first-level EC labels collapse substantial biological diversity.

### Results
Report the measured test results from the final summary table rather than copying example numbers from the project description.

### Conclusion
State which representation and ensemble produced the observed validation/test behavior, whether ensemble learning improved over the individual models, which classes were most difficult, and what the next methodological improvement would be.

# 31. References

1. **Project specification provided for this assignment.** The supplied document defines the SwissProt-EC dataset, EC 1–6 target, feature representations, leakage controls, models, ensembles, metrics, error analysis and optional SHAP/SVD tracks.
2. DanielHesslow, **SwissProt-EC**, Hugging Face dataset, as specified by the project document.
3. Scikit-learn documentation for `CountVectorizer`, `SelectKBest`, `TruncatedSVD`, `RandomForestClassifier`, `SVC`, `VotingClassifier`, and `StackingClassifier`.
4. LightGBM documentation for gradient-boosted decision trees.
5. SHAP documentation for tree-based model explanations.

> **Interpretation rule:** empirical findings in this notebook should be described as observations from this dataset and evaluation split. Biological explanations should be presented as hypotheses unless supported by an independent biological analysis.

## Save the trained model
The selected estimator and the feature transforms are saved together for local reuse.


In [ ]:
import joblib
artifact = {
    "model": FINAL_MODEL,
    "model_name": FINAL_MODEL_NAME,
    "representation": SELECTED_REPRESENTATION,
    "class_order": CLASS_ORDER.tolist(),
    "amino_acids": AMINO_ACIDS,
    "dipeptide_vectorizer": di_vec if SELECTED_REPRESENTATION == "Dipeptide" else None,
    "tripeptide_vectorizer": tri_vec if SELECTED_REPRESENTATION == "Tripeptide-Selected" else None,
    "tripeptide_selector": tri_selector if SELECTED_REPRESENTATION == "Tripeptide-Selected" else None,
    "random_state": RANDOM_STATE,
}
model_path = OUTPUT_DIR / "enzyme_function_classifier.joblib"
joblib.dump(artifact, model_path, compress=3)
test_results.to_csv(OUTPUT_DIR / "test_results.csv", index=False)
print(f"Saved model and feature transforms to: {model_path.resolve()}")
print(f"Saved test metrics to: {(OUTPUT_DIR / 'test_results.csv').resolve()}")
